# 06 — Weather and arrival efficiency (Gold consumer)

**Question.** *Does the weather at the destination airport explain part of the time lost on arrival, and does it
change the conclusions of notebook 05 (Heathrow saturated by demand, level flight at Paris-CDG and Frankfurt)?*

**Data.** METAR (airport weather reports, every 30 or 60 minutes) of the real period of the scope, from the public
archive of the Iowa Environmental Mesonet, for 34 airports: the 30 airports with the most measured arrivals in prod-8
and the focus airports outside that list. They go through the same layers as OPDI: raw files in the manifest, Bronze
with lineage and a schema contract, Silver with quality rules and quarantine (`silver/weather_observations`), Gold
(`airport_weather_hourly`, `arrival_weather_impact`), and the ML layer (`ml_weather_comparison`).

**Weather category of an hour** (most severe first; thresholds in `config/prod.toml`, `[weather]`):
`THUNDERSTORM` · `LOW_VISIBILITY` (below the ICAO CAT I minima: visibility < 800 m or ceiling < 200 ft) ·
`IMC` (instrument conditions: visibility < 5 km or ceiling < 1 500 ft, ICAO VFR minima) · `STRONG_WIND` (≥ 20 kt or
gusts ≥ 30 kt) · `PRECIPITATION` · `GOOD`.

**Limits.** The weather is the one of the arrival hour at the destination only (not en route, not upper winds);
summer period (few low-visibility hours); associations, not causality — a stormy hour can also be a busy hour.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))      # projet_opdi/src
import opdi_pipeline                                     # first: on the VM, findspark points to Spark 3.5 (/opt/spark)
from opdi_pipeline.config import load_settings
from opdi_pipeline.spark import get_spark

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"axes.edgecolor": "#cccccc", "axes.linewidth": 0.8, "figure.dpi": 110})

ENV = "prod"                     # "prod": full scope (~/opdi-data) | "dev": 10-day sample (~/opdi-data-dev)
settings = load_settings(ENV)
spark = get_spark(settings)
FIG_DIR = Path.cwd().parent / "docs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

FOCUS = ["LFPG", "LFBO", "ENZV", "EGLL", "EDDF"]
CATEGORIES = ["GOOD", "PRECIPITATION", "STRONG_WIND", "IMC", "LOW_VISIBILITY", "THUNDERSTORM"]
CAT_COLORS = dict(zip(CATEGORIES, ["#8fbf8f", "#6baed6", "#fdae6b", "#9e9ac8", "#636363", "#d62728"]))
MIN_MEASURED = 100 if ENV == "prod" else 5     # indicator A of a group only shown above this number of measured arrivals
print(f"{ENV}: scope [{settings.start}, {settings.end}[ | {len(settings.weather.stations)} weather stations")

## Gold tables and run reports

In [ ]:
assert (settings.gold_dir / "_SUCCESS").exists(), "Gold is not published: the last run did not pass its checks"
g = settings.gold_dir
read = lambda name: spark.read.parquet(str(g / name)).toPandas() if (g / name).exists() else pd.DataFrame()
wh, impact, hourly = read("airport_weather_hourly"), read("arrival_weather_impact"), read("airport_arrival_hourly")
comparison, wfi = read("ml_weather_comparison"), read("ml_weather_feature_importance")
assert not wh.empty, "no weather table: check the weather part of the run summary"

def last_report(layer):
    paths = sorted((settings.data_root / "reports").glob(f"*/{layer}.json"), key=lambda p: p.stat().st_mtime)
    return json.loads(paths[-1].read_text()) if paths else {}

silver_report, ml_report = last_report("silver"), last_report("ml")
w = silver_report.get("weather", {})
print(f"{w.get('accounting', {}).get('silver', 0):,} METAR in Silver | stations without data: "
      f"{w.get('stations_without_data')} | quarantine: {w.get('quarantine_reasons')}")
print(f"{len(wh):,} airport-hours with weather ({wh['airport_icao'].nunique()} airports) | run {ml_report.get('run_id')}")

## 1. How often is the weather bad?

Share of the hours of each airport in each weather category (summer 2026). Silver row accounting of the weather is in
the Silver report (`weather.accounting`).

In [ ]:
share = (wh.groupby(["airport_icao", "weather_category"]).size().unstack(fill_value=0)
         .reindex(columns=CATEGORIES, fill_value=0))
share = share.div(share.sum(axis=1), axis=0) * 100
share = share.sort_values("GOOD")
fig, ax = plt.subplots(figsize=(13, 6))
left = np.zeros(len(share))
for cat in CATEGORIES:
    ax.barh(share.index, share[cat], left=left, color=CAT_COLORS[cat], label=cat)
    left += share[cat].values
for label in ax.get_yticklabels():
    if label.get_text() in FOCUS:
        label.set_fontweight("bold")
ax.set_xlabel("share of the hours (%)"); ax.set_xlim(0, 100)
ax.legend(ncol=6, loc="upper center", bbox_to_anchor=(0.5, 1.08), fontsize=9, frameon=False)
ax.set_title("Weather of the hours, by airport", fontweight="bold", pad=28)
plt.tight_layout(); save(fig, "08_weather_conditions"); plt.show()
share.loc[[a for a in FOCUS if a in share.index]].round(1)

## 2. Arrival indicators by weather category

All airports with weather together, complete traffic hours. Additional time (indicator A) weighted by the measured
arrivals; level-off minutes and holdings (indicator B) weighted by all arrivals. Categories with fewer than
`MIN_MEASURED` measured arrivals are not shown for indicator A.

In [ ]:
def pooled(df):
    return pd.Series({
        "hours": df["hours"].sum(), "arrivals": df["arrivals"].sum(), "measured": df["arrivals_measured"].sum(),
        "additional_min": np.average(df["avg_additional_min"].fillna(0), weights=df["arrivals_measured"])
                          if df["arrivals_measured"].sum() else np.nan,
        "level_min": np.average(df["avg_level_minutes"], weights=df["arrivals"]),
        "pct_holding": np.average(df["pct_with_holding"], weights=df["arrivals"])})

by_cat = impact.groupby("weather_category").apply(pooled).reindex(CATEGORIES).dropna(how="all")
by_cat.loc[by_cat["measured"] < MIN_MEASURED, "additional_min"] = np.nan
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, colname, title in zip(axes, ["additional_min", "level_min", "pct_holding"],
                              ["Additional arrival time (min / flight)", "Level-off minutes per arrival",
                               "Arrivals with a holding (%)"]):
    ax.bar(by_cat.index, by_cat[colname], color=[CAT_COLORS[c] for c in by_cat.index])
    good = by_cat.loc["GOOD", colname] if "GOOD" in by_cat.index else np.nan
    ax.axhline(good, color="#555555", ls="--", lw=1)
    ax.set_title(title, fontweight="bold", fontsize=11); ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); save(fig, "09_weather_impact"); plt.show()
by_cat.round(2)

Per focus airport (rows with too few measured arrivals for indicator A are blank):

In [ ]:
rows = []
for (apt, cat), d in impact[impact["airport_icao"].isin(FOCUS)].groupby(["airport_icao", "weather_category"]):
    r = pooled(d)
    if r["measured"] < MIN_MEASURED:
        r["additional_min"] = np.nan
    rows.append(r.rename((apt, cat)))
focus = pd.DataFrame(rows)
focus.index = pd.MultiIndex.from_tuples(focus.index, names=["airport", "weather"])
focus.round(2)

## 3. Weather or demand?

A stormy hour can also be a busy hour. For the three hubs, the additional time by number of arrivals in the hour
(bins of 10), separately for the hours with good weather and the hours with adverse weather (any other category).
If the two curves stay close, the demand effect seen in notebook 05 is not a weather effect in disguise.

In [ ]:
hw = hourly.merge(wh[["airport_icao", "weather_date", "weather_hour", "weather_category"]],
                  left_on=["airport_icao", "movement_date", "movement_hour"],
                  right_on=["airport_icao", "weather_date", "weather_hour"], how="inner")
hw = hw[hw["is_complete_hour"] & (hw["arrivals_measured"] > 0)].copy()
hw["weather"] = np.where(hw["weather_category"] == "GOOD", "good weather", "adverse weather")
hw["bin"] = (hw["arrivals"] // 10) * 10 + 5
hubs = [a for a in ["EGLL", "LFPG", "EDDF"] if a in set(hw["airport_icao"])]
fig, axes = plt.subplots(1, max(len(hubs), 1), figsize=(16, 4.8), sharey=True, squeeze=False)
for ax, apt in zip(axes[0], hubs):
    for label, colour in [("good weather", "#2ca02c"), ("adverse weather", "#d62728")]:
        d = hw[(hw["airport_icao"] == apt) & (hw["weather"] == label)]
        b = d.groupby("bin").apply(lambda z: pd.Series({
            "mean": np.average(z["avg_additional_min"], weights=z["arrivals_measured"]),
            "n": z["arrivals_measured"].sum()}))
        b = b[b["n"] >= 30] if not b.empty else b
        if not b.empty:
            ax.plot(b.index, b["mean"], "-o", ms=3, color=colour, label=label)
    ax.set_title(apt, fontweight="bold"); ax.set_xlabel("arrivals in the hour")
axes[0][0].set_ylabel("additional arrival time (min / flight)"); axes[0][0].legend()
plt.tight_layout(); save(fig, "10_weather_or_demand"); plt.show()
hw.groupby(["airport_icao", "weather"])["arrivals"].agg(["count", "mean"]).loc[hubs].round(1)

## 4. Does the weather improve the model?

The gradient-boosted trees of notebook 05 (same features, same hyperparameters, same temporal split), trained on the
same arrivals (airports with METAR) **without** and **with** the weather of the arrival hour, both compared with the
naive baseline (mean per airport and hour) on the same test days. The importance of the weather variables is only
published if the model with weather beats the naive baseline.

In [ ]:
wm = ml_report.get("weather", {})
print("status:", wm.get("status"), "| MAE gain (min):", wm.get("delta_mae_min"), "| R2 gain:", wm.get("delta_r2"),
      "| weather share of the importance:", wm.get("weather_importance_share"))
if not wfi.empty:
    weather_vars = {"avg_wind_speed_kt", "max_gust_kt", "visibility_km", "ceiling_kft", "thunderstorm",
                    "precipitation", "fog", "imc", "below_cat1"}
    top = wfi.sort_values("importance").tail(12)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top["variable"], top["importance"],
            color=["#d62728" if v in weather_vars else "#4c72b0" for v in top["variable"]])
    ax.set_xlabel("relative importance"); ax.set_title("Model with weather: variable importance (weather in red)",
                                                       fontweight="bold")
    plt.tight_layout(); save(fig, "11_weather_model"); plt.show()
comparison.drop(columns=["_ml_built_at_utc"], errors="ignore").round(3)

## 5. Reading the figures

*To be written from the production run.*